In [10]:
# ============================================================
# MODULE 4.1 — PDF LOGICAL SPLITTING
# Boundary Detection Pipeline
# ============================================================

# ── CELL 1: CÀI ĐẶT ─────────────────────────────────────────
!pip install pymupdf pdfplumber pdf2image pytesseract -q
!apt-get install -y tesseract-ocr tesseract-ocr-vie -q

import fitz, pdfplumber, re, json
import pytesseract
from pdf2image import convert_from_path
from google.colab import drive

print("✅ Setup xong!")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-vie is already the newest version (1:4.00~git30-7274cfa-1.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
✅ Setup xong!


In [11]:
# ── CELL 2: CONFIG ───────────────────────────────────────────
PDF_PATH = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/mix_doc/document_merged2.pdf'

RE_SO_VAN_BAN = r'\d{1,3}/\d{4}/[A-ZĐQĐ][\w\-]+'

# ✅ FIX: trang 34 mới là blank thật (char=0), trang 3 có nội dung
GT_BOUNDARIES = {1, 4, 7, 9, 29, 35, 43, 44, 50, 52, 54, 57, 59, 62, 63, 66, 75, 81, 103, 105, 108}
GT_SEPARATORS = {65}

# Ground truth loại tài liệu — dựa trên header thực tế
GT_SEGMENT_TYPES = {
    1 : "Bản án hành chính",   # 427/2019/HC-PT — TÒA ÁN CẤP CAO TPHCM
    23: "Bản án dân sự",       # 03/2017/DS-ST
    29: "Quyết định",          # 140/2025/QĐST-HC — Quyết định đình chỉ
    32: "Quyết định",          # 01/2026/QĐ-PT
    35: "Bản án hình sự",      # 04/2017/HSST
    44: "Bản án hình sự",      # 170/2026/HS-PT (từ ảnh upload)
}

with fitz.open(PDF_PATH) as doc:
    TOTAL_PAGES = len(doc)

print(f"✅ Config xong | {PDF_PATH.split('/')[-1]} | {TOTAL_PAGES} trang")
print(f"   GT_BOUNDARIES : {sorted(GT_BOUNDARIES)}")
print(f"   GT_SEPARATORS : {sorted(GT_SEPARATORS)}")

✅ Config xong | document_merged2.pdf | 109 trang
   GT_BOUNDARIES : [1, 4, 7, 9, 29, 35, 43, 44, 50, 52, 54, 57, 59, 62, 63, 66, 75, 81, 103, 105, 108]
   GT_SEPARATORS : [65]


In [12]:
# ── CELL 3: EXTRACT TEXT & FEATURES ────────────────────────
import io
from PIL import Image

def get_text_hybrid_fast(pdf_path: str, page_idx: int) -> str:
    with fitz.open(pdf_path) as doc:
        page = doc[page_idx]
        text = page.get_text("text").strip()

        # Nếu không có text gốc (ảnh scan), chuyển sang OCR tốc độ cao
        if len(text) < 10:
            pix = page.get_pixmap(dpi=150, colorspace=fitz.csGRAY)
            img = Image.frombytes("L", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(img, lang='vie').strip()

    return text


def extract_features(pdf_path: str, page_idx: int) -> dict:
    # ĐÃ SỬA LỖI: Gọi đúng hàm hybrid OCR
    text = get_text_hybrid_fast(pdf_path, page_idx)
    lines = text.split("\n") if text else []
    n = len(lines)

    # Gom 20% số dòng đầu tiên làm Header (tối thiểu 5 dòng) để định vị
    header = " ".join(l.strip() for l in lines[:max(5, int(n * 0.20))] if l.strip())
    header_up = header.upper()

    # 1. Tìm Quốc hiệu / Tiêu ngữ / Tên cơ quan đặc trưng TẠI HEADER
    has_quoc_hieu = "CỘNG HÒA XÃ HỘI" in header_up
    has_tieu_ngu = "HẠNH PHÚC" in header_up
    has_toa_an = "TÒA ÁN NHÂN DÂN" in header_up

    # 2. Bắt Số văn bản, nhưng CHỈ BẮT Ở HEADER để tránh dính trích dẫn
    so_vb_header = re.findall(RE_SO_VAN_BAN, header)

    return {
        "page_num": page_idx + 1,
        "char_count": len(text),
        "is_blank": len(text) < 30,
        "header": header,
        "has_quoc_hieu": has_quoc_hieu,
        "has_tieu_ngu": has_tieu_ngu,
        "has_toa_an": has_toa_an,
        "so_vb_header": so_vb_header,
        "text_full": text,
    }

# Chạy extract
print("⏳ Đang extract features (sử dụng Hybrid OCR)...\n")
print(f"{'Trang':<7}{'Ký tự':<8}{'Blank':<7}{'Quốc Hiệu':<12}{'Tòa Án':<10}{'Số VB (Header)'}")
print("─" * 75)

all_features = []
for i in range(TOTAL_PAGES):
    f = extract_features(PDF_PATH, i)
    all_features.append(f)

    so_vb_str = f["so_vb_header"][0] if f["so_vb_header"] else "—"
    qh_str = "✅" if f['has_quoc_hieu'] else "—"
    ta_str = "✅" if f['has_toa_an'] else "—"

    print(
        f"  {f['page_num']:<5}"
        f"{f['char_count']:<8}"
        f"{'🔴' if f['is_blank'] else '':<7}"
        f"{qh_str:<12}"
        f"{ta_str:<10}"
        f"{so_vb_str}"
    )

print("─" * 75)
print(f"✅ Extract xong {len(all_features)} trang!")

⏳ Đang extract features (sử dụng Hybrid OCR)...

Trang  Ký tự   Blank  Quốc Hiệu   Tòa Án    Số VB (Header)
───────────────────────────────────────────────────────────────────────────
  1    2165           ✅           ✅         200/2026/QĐ-PT
  2    2068           —           —         —
  3    880            —           ✅         82/2025/KDTM-ST
  4    2053           ✅           ✅         14/2026/QĐ-PT
  5    2364           —           —         —
  6    246            —           —         —
  7    1613           —           ✅         81/2026/QĐ-PT
  8    373            —           —         —
  9    1706           —           ✅         453/2026/DS-PT
  10   2007           —           —         —
  11   2837           —           —         —
  12   2890           —           —         —
  13   2549           —           —         —
  14   2564           —           —         —
  15   2755           —           —         —
  16   2673           —           —         —
  17   2533     

In [13]:
def extract_strict_header(text: str, n_lines: int = 5) -> tuple[str, str]:
    """
    Trả về:
      strict_header : 5 dòng đầu có nội dung (bỏ số trang đơn)
      first_line    : dòng đầu tiên thật sự (quan trọng để kiểm tra)
    """
    lines = text.split("\n") if text else []
    result = []
    first_line = ""

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        # Bỏ dòng chỉ là số trang đơn lẻ (1–3 chữ số)
        if re.match(r'^\d{1,3}$', stripped):
            continue
        if not first_line:
            first_line = stripped
        result.append(stripped)
        if len(result) >= n_lines:
            break

    return " ".join(result), first_line


# Cờ nhận diện: dòng ĐẦU TIÊN phải là tên cơ quan hoặc quốc hiệu
FIRST_LINE_PATTERNS = [
    r'TÒA ÁN NHÂN DÂN',
    r'VIỆN KIỂM SÁT',
    r'CỘNG HÒA XÃ HỘI',
    r'NHÂN DANH',
    r'QUYẾT ĐỊNH\s*$',     # "QUYẾT ĐỊNH" đứng cuối dòng một mình
]

def is_doc_start_header(strict_header: str, first_line: str) -> tuple[bool, str]:
    """
    Trang đầu tài liệu mới phải thỏa MẶT CẢ 2 điều kiện:
      1. Dòng đầu tiên khớp với pattern cơ quan / quốc hiệu
      2. Trong 5 dòng đầu có đủ 2/3 nhóm signal

    Trang nội dung: dòng đầu là text chạy tiếp ("phí, lệ phí theo...")
    → Không thỏa điều kiện 1 → loại ngay, không cần check tiếp.
    """
    h          = strict_header.upper()
    first_up   = first_line.upper()

    # ── Điều kiện 1: dòng đầu phải là cơ quan / quốc hiệu ───
    first_line_ok = any(re.search(p, first_up) for p in FIRST_LINE_PATTERNS)
    if not first_line_ok:
        return False, ""   # ← loại ngay, trang 31 bị chặn ở đây

    # ── Điều kiện 2: đủ 2/3 nhóm signal trong 5 dòng đầu ───
    group_a = ("CỘNG HÒA XÃ HỘI" in h or "ĐỘC LẬP" in h or "HẠNH PHÚC" in h)
    group_b = ("TÒA ÁN NHÂN DÂN" in h or "VIỆN KIỂM SÁT" in h)
    group_c = (
        bool(re.search(r'\d{1,3}/\d{4}/[A-ZĐQĐ][\w\-]+', strict_header))
        or "BẢN ÁN SỐ" in h
        or "NHÂN DANH" in h
        or bool(re.search(r'QUYẾT ĐỊNH\s*$', h, re.MULTILINE))
    )

    groups_met = sum([group_a, group_b, group_c])
    if groups_met >= 2:
        signals = []
        if group_a: signals.append("quoc_hieu")
        if group_b: signals.append("co_quan")
        if group_c: signals.append("dinh_danh")
        return True, "+".join(signals)

    return False, ""


def is_boundary(feat: dict, prev: dict | None) -> tuple[bool, str, float]:
    if prev is None:
        return True, "first_page", 1.0
    if feat["is_blank"]:
        return False, "blank_page", 0.0
    if prev["is_blank"]:
        return True, "after_blank", 0.95

    # Dùng strict_header + first_line
    strict_hdr, first_line = feat["strict_header"]   # giờ là tuple
    is_start, signal = is_doc_start_header(strict_hdr, first_line)
    if is_start:
        conf = 1.0 if "quoc_hieu" in signal and "co_quan" in signal else 0.9
        return True, signal, conf

    return False, "continuation", 0.0


# ── Cập nhật all_features với strict_header mới (tuple) ──────
for f in all_features:
    f["strict_header"] = extract_strict_header(f["text_full"], n_lines=5)
    # f["strict_header"] giờ là tuple (header_str, first_line_str)


# ── Chạy lại Boundary Detection ──────────────────────────────
print("BOUNDARY DETECTION RESULTS (STRICT HEADER v2 — fix FP trang 31)")
print("=" * 110)
print(f"{'Trang':<7}{'Predict':<13}{'GT':<13}{'Match':<9}{'Conf':<7}{'Signal':<32}{'First line (40 ký tự)'}")
print("─" * 110)

predict_set = set()
for i, feat in enumerate(all_features):
    prev              = all_features[i-1] if i > 0 else None
    bdry, reason, conf = is_boundary(feat, prev)
    pn                = feat["page_num"]

    if bdry:
        predict_set.add(pn)

    gt_label   = "BOUNDARY"   if pn in GT_BOUNDARIES else \
                 ("SEPARATOR" if pn in GT_SEPARATORS else "continuation")
    pred_label = "BOUNDARY"   if bdry else \
                 ("separator" if feat["is_blank"] else "continuation")
    match      = ("✅ HIT" if bdry else "❌ MISS") if pn in GT_BOUNDARIES else \
                 ("⚠️ FP"  if bdry else "")        if pn not in GT_SEPARATORS else ""

    if match or pn in GT_BOUNDARIES | GT_SEPARATORS or bdry:
        # first_line từ tuple
        first_ln = feat["strict_header"][1][:40] if isinstance(feat["strict_header"], tuple) else ""
        print(f"  {pn:<5}{pred_label:<13}{gt_label:<13}{match:<9}{conf:<7}{reason:<32}{first_ln}")

print("=" * 110)

# Metrics ngay
TP = len(predict_set & GT_BOUNDARIES)
FP = len(predict_set - GT_BOUNDARIES)
FN = len(GT_BOUNDARIES - predict_set)
p  = TP / (TP + FP) if (TP + FP) > 0 else 0
r  = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2*p*r/(p+r)    if (p+r)    > 0 else 0
print(f"\n📊 F1={f1:.3f} | Precision={p:.3f} | Recall={r:.3f} | TP={TP} FP={FP} FN={FN}")
print(f"   {'✅ Đạt F1 ≥ 0.90' if f1 >= 0.9 else '⚠️ Chưa đạt'}")

BOUNDARY DETECTION RESULTS (STRICT HEADER v2 — fix FP trang 31)
Trang  Predict      GT           Match    Conf   Signal                          First line (40 ký tự)
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1    BOUNDARY     BOUNDARY     ✅ HIT    1.0    first_page                      TÒA ÁN NHÂN DÂN
  4    BOUNDARY     BOUNDARY     ✅ HIT    1.0    quoc_hieu+co_quan+dinh_danh     TÒA ÁN NHÂN DÂN CỘNG HÒA XÃ HỘI CHỦ NGHĨ
  7    BOUNDARY     BOUNDARY     ✅ HIT    1.0    quoc_hieu+co_quan+dinh_danh     TÒA ÁN NHÂN DÂN CỌNG HÒA XÃ HỌI CHỦ NGHĨ
  9    BOUNDARY     BOUNDARY     ✅ HIT    0.9    co_quan+dinh_danh               TÒA ÁN NHÂN DÂN
  29   BOUNDARY     BOUNDARY     ✅ HIT    1.0    quoc_hieu+co_quan+dinh_danh     TÒA ÁN NHÂN DÂN
  35   BOUNDARY     BOUNDARY     ✅ HIT    1.0    quoc_hieu+co_quan+dinh_danh     TÒA ÁN NHÂN DÂN CỘNG HÒA XÃ HỌI CHỦ NGHĨ
  43   BOUNDARY     BOUNDARY     ✅ HIT    1.0    quoc_hieu+co_qua

In [14]:
# ── CELL 5: METRICS ──────────────────────────────────────────
TP = len(predict_set & GT_BOUNDARIES)
FP = len(predict_set - GT_BOUNDARIES)
FN = len(GT_BOUNDARIES - predict_set)

p  = TP / (TP + FP) if (TP + FP) > 0 else 0
r  = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2*p*r / (p+r)  if (p + r)   > 0 else 0

print("📊 KẾT QUẢ")
print("=" * 50)
print(f"  GT predict    : {sorted(GT_BOUNDARIES)}")
print(f"  Predict       : {sorted(predict_set)}")
print(f"  TP / FP / FN  : {TP} / {FP} / {FN}")
print(f"  Precision     : {p:.3f}")
print(f"  Recall        : {r:.3f}")
print(f"  F1 Score      : {f1:.3f}  {'✅ Đạt' if f1 >= 0.90 else '⚠️ Chưa đạt 0.90'}")
print("=" * 50)

for pn in sorted(GT_BOUNDARIES - predict_set):
    f = all_features[pn-1]
    print(f"\n❌ Miss trang {pn}: Q.Hiệu={f['has_quoc_hieu']} | T.Án={f['has_toa_an']} | Số VB={f['so_vb_header']}")
    print(f"   Header: {f['header'][:120]}")

for pn in sorted(predict_set - GT_BOUNDARIES):
    f = all_features[pn-1]
    print(f"\n⚠️  FP  trang {pn}: Q.Hiệu={f['has_quoc_hieu']} | T.Án={f['has_toa_an']} | Số VB={f['so_vb_header']}")
    print(f"   Header: {f['header'][:120]}")

📊 KẾT QUẢ
  GT predict    : [1, 4, 7, 9, 29, 35, 43, 44, 50, 52, 54, 57, 59, 62, 63, 66, 75, 81, 103, 105, 108]
  Predict       : [1, 4, 7, 9, 29, 35, 43, 44, 50, 52, 54, 57, 59, 62, 63, 66, 75, 81, 103, 105, 108]
  TP / FP / FN  : 21 / 0 / 0
  Precision     : 1.000
  Recall        : 1.000
  F1 Score      : 1.000  ✅ Đạt


In [15]:
# ── CELL 6: VERIFY GT + ĐÁNH DẤU START/END TỪNG TÀI LIỆU ────

# ── Phần 1: Metrics boundary ─────────────────────────────────
TP = len(predict_set & GT_BOUNDARIES)
FP = len(predict_set - GT_BOUNDARIES)
FN = len(GT_BOUNDARIES - predict_set)
p  = TP / (TP + FP) if (TP + FP) > 0 else 0
r  = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2*p*r / (p+r)  if (p + r) > 0 else 0

print("📊 BOUNDARY METRICS")
print(f"  Predict : {sorted(predict_set)}")
print(f"  GT      : {sorted(GT_BOUNDARIES)}")
print(f"  TP/FP/FN: {TP} / {FP} / {FN}")
print(f"  F1      : {f1:.3f}  {'✅ Đạt' if f1 >= 0.9 else '⚠️ Chưa đạt 0.90'}")

# ── Phần 2: Đánh dấu START / END / SEPARATOR từng tài liệu ───
print("=" * 75)
print("CHI TIẾT TỪNG TRANG — START / END / SEPARATOR")
print("=" * 75)

# Xây dựng boundary list từ predict_set
boundary_pages = sorted(predict_set)

# Gom thành đoạn [start, end]
doc_ranges = []
for idx, start in enumerate(boundary_pages):
    # Tìm trang kết thúc = trước boundary tiếp theo hoặc trước separator hoặc hết file
    if idx + 1 < len(boundary_pages):
        next_start = boundary_pages[idx + 1]
        # End = trang ngay trước next_start (bỏ qua separator ở giữa)
        end = next_start - 1
        # Lùi qua separator nếu có
        while end in GT_SEPARATORS:
            end -= 1
    else:
        end = TOTAL_PAGES
        while end in GT_SEPARATORS:
            end -= 1
    doc_ranges.append((start, end))

# In bảng đánh dấu
for pn in range(1, TOTAL_PAGES + 1):
    feat = all_features[pn - 1]

    # Xác định vai trò trang
    if pn in GT_SEPARATORS or feat["is_blank"]:
        role = "── SEPARATOR ──"
        tag  = "🔴"
    else:
        # Tìm document range chứa trang này
        role = "continuation"
        tag  = "   "
        for (s, e) in doc_ranges:
            if s <= pn <= e:
                if pn == s:
                    gt_type = GT_SEGMENT_TYPES.get(s, "?")
                    role = f"▶ START  [{gt_type}]"
                    tag  = "🟢"
                elif pn == e:
                    role = "◀ END"
                    tag  = "🔵"
                else:
                    role = "│ nội dung"
                    tag  = "   "
                break

    # Access the string part of the tuple before applying string methods
    hdr_content = feat["strict_header"][0] if isinstance(feat["strict_header"], tuple) else feat["strict_header"]
    hdr = hdr_content[:55].replace("\n"," ") if not feat["is_blank"] else "(trống)"
    print(f"  {tag} Trang {pn:02d} | {role:<35} | {hdr}")

print("=" * 75)
print(f"\n📄 Tổng: {len(doc_ranges)} tài liệu, {len(GT_SEPARATORS)} trang separator")
for i, (s, e) in enumerate(doc_ranges):
    gt = GT_SEGMENT_TYPES.get(s, "?")
    print(f"  Tài liệu {i+1}: trang {s:>2}–{e:>2} ({e-s+1:>2} trang) — {gt}")

📊 BOUNDARY METRICS
  Predict : [1, 4, 7, 9, 29, 35, 43, 44, 50, 52, 54, 57, 59, 62, 63, 66, 75, 81, 103, 105, 108]
  GT      : [1, 4, 7, 9, 29, 35, 43, 44, 50, 52, 54, 57, 59, 62, 63, 66, 75, 81, 103, 105, 108]
  TP/FP/FN: 21 / 0 / 0
  F1      : 1.000  ✅ Đạt
CHI TIẾT TỪNG TRANG — START / END / SEPARATOR
  🟢 Trang 01 | ▶ START  [Bản án hành chính]        | TÒA ÁN NHÂN DÂN THÀNH PHỐ HỒ CHÍ MINH Số: 200/2026/QĐ-P
      Trang 02 | │ nội dung                          | Ý kiến của nguyên đơn và đại diện Viện kiểm sát nhân dâ
  🔵 Trang 03 | ◀ END                               | 2. Giữ nguyên Bản án sơ thẩm số 82/2025/KDTM-ST ngày 23
  🟢 Trang 04 | ▶ START  [?]                        | TÒA ÁN NHÂN DÂN CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM TỈNH
      Trang 05 | │ nội dung                          | 036146001147; Địa chỉ đăng ký thường trú: Số I ô 19 phư
  🔵 Trang 06 | ◀ END                               | 4. Quyết định đình chỉ xét xử phúc thẩm vụ án dân sự có
  🟢 Trang 07 | ▶ START  [?]          

In [16]:
# ── CELL 7: STAGE 3 — SEGMENT CLASSIFICATION (FIX FINAL) ─────
#
# Chiến lược mới: ưu tiên số ký hiệu văn bản trong header làm signal mạnh nhất
# vì ký hiệu rất đặc trưng và không bị overlap:
#   HC-PT / HC-ST  → Bản án hành chính
#   HS-PT / HS-ST / HSPT / HSST → Bản án hình sự
#   DS-PT / DS-ST  → Bản án dân sự
#   QĐ-PT / QĐST  / QĐ  → Quyết định

import re

# Regex nhận diện ký hiệu loại văn bản trong số văn bản
SO_VB_TYPE_PATTERNS = {
    "Bản án hành chính": [r'HC-PT', r'HC-ST', r'/HC\b'],
    "Bản án hình sự":    [r'HS-PT', r'HS-ST', r'HSPT', r'HSST', r'/HS\b'],
    "Bản án dân sự":     [r'DS-PT', r'DS-ST', r'DSPT', r'DSST', r'/DS\b'],
    "Quyết định":        [r'QĐ-PT', r'QĐST', r'QĐ-UBND', r'QĐ-TAND',
                          r'QĐ-HC', r'QĐ\b'],
}

# Keyword exclusive — mỗi keyword chỉ xuất hiện ở 1 loại
SEGMENT_KEYWORDS = {
    "Bản án hành chính": [
        "người khởi kiện",
        "người bị kiện",
        "khiếu kiện quyết định hành chính",
        "khởi kiện vụ án hành chính",
        "xét xử vụ án hành chính",
    ],
    "Bản án hình sự": [
        "bị cáo",
        "viện kiểm sát nhân dân",
        "phạm tội",
        "phạt tù",
        "tuyên phạt",
    ],
    "Bản án dân sự": [
        "nguyên đơn",
        "bị đơn",
        "tranh chấp",
        "thừa kế",
        "hôn nhân gia đình",
    ],
    "Quyết định": [
        "đình chỉ giải quyết vụ án",
        "hủy bản án sơ thẩm",
        "nơi nhận:",
        "tạm ứng án phí",
        "người có quyền lợi nghĩa vụ liên quan",
    ],
}

STRONG_TYPE_KW = {
    "Bản án hành chính": ["bản án hành chính", "vụ án hành chính"],
    "Bản án hình sự":    ["bản án hình sự", "vụ án hình sự"],
    "Bản án dân sự":     ["bản án dân sự", "vụ án dân sự"],
    "Quyết định":        ["quyết định đình chỉ", "quyết định hủy bản án",
                          "đình chỉ giải quyết vụ án hành chính"],
}

GT_SEGMENT_TYPES = {
    1 : "Bản án hành chính",
    23: "Bản án dân sự",
    29: "Quyết định",
    32: "Quyết định",
    35: "Bản án hình sự",
    44: "Bản án hình sự",   # 170/2026/HS-PT từ ảnh
}


def classify_by_so_vb(so_vb_list: list) -> str | None:
    """
    Classify dựa trên ký hiệu số văn bản trong header trang đầu.
    Đây là signal mạnh nhất, ưu tiên trước keyword voting.
    """
    for so_vb in so_vb_list:
        so_vb_upper = so_vb.upper()
        for label, patterns in SO_VB_TYPE_PATTERNS.items():
            for pat in patterns:
                if re.search(pat, so_vb_upper):
                    return label
    return None


def classify_segment(pages: list) -> tuple[str, float, dict]:
    """
    Phân loại segment theo 3 bước ưu tiên:
      1. Số ký hiệu văn bản của trang ĐẦU → chính xác nhất
      2. Strong keyword trong toàn bộ text
      3. Exclusive keyword voting
    """
    full_text    = " ".join(f["text_full"] for f in pages)
    full_lower   = full_text.lower()
    header_lower = pages[0]["header"].lower() if pages else ""

    # Lấy strict_header tuple từ trang đầu
    sh = pages[0].get("strict_header", ("", ""))
    strict_str   = sh[0].lower() if isinstance(sh, tuple) else str(sh).lower()

    scores = {label: 0.0 for label in SEGMENT_KEYWORDS}

    # ── Bước 1: Classify bằng số ký hiệu văn bản (header trang đầu) ──
    # Lấy all so_vb trong strict header trang đầu
    so_vb_in_header = re.findall(RE_SO_VAN_BAN, pages[0].get("header", ""))
    label_from_so_vb = classify_by_so_vb(so_vb_in_header)

    if label_from_so_vb:
        # Signal mạnh nhất — cộng điểm rất cao
        scores[label_from_so_vb] += 5.0
        conf_boost = True
    else:
        conf_boost = False

    # ── Bước 2: Strong keyword toàn text ──────────────────────
    for label, kws in STRONG_TYPE_KW.items():
        for kw in kws:
            if kw in full_lower or kw in strict_str:
                scores[label] += 3.0

    # ── Bước 3: Exclusive keyword voting ──────────────────────
    for label, kws in SEGMENT_KEYWORDS.items():
        hits = sum(1 for kw in kws if kw in full_lower)
        scores[label] += hits / len(kws)

    # ── Header boost x1.5 ─────────────────────────────────────
    for label, kws in SEGMENT_KEYWORDS.items():
        header_hits = sum(1 for kw in kws if kw in header_lower)
        scores[label] += (header_hits / len(kws)) * 1.5

    # ── Tính confidence ───────────────────────────────────────
    best_label = max(scores, key=scores.get)
    total      = sum(scores.values())
    confidence = round(scores[best_label] / total, 3) if total > 0 else 0.0

    if total == 0:
        best_label = "Không xác định"
        confidence = 0.0

    return best_label, confidence, {k: round(v, 3) for k, v in scores.items()}


# ── Gom segments từ predict_set ──────────────────────────────
def group_segments(all_features: list, boundary_pages: set,
                   separator_pages: set) -> list:
    segments  = []
    current   = []
    for f in all_features:
        pn = f["page_num"]
        if f["is_blank"] or pn in separator_pages:
            if current:
                segments.append(current)
                current = []
            continue
        if pn in boundary_pages and current:
            segments.append(current)
            current = []
        current.append(f)
    if current:
        segments.append(current)
    return segments


segments = group_segments(all_features, predict_set, GT_SEPARATORS)

# ── Chạy classify ─────────────────────────────────────────────
print("SEGMENT CLASSIFICATION RESULTS")
print("=" * 95)
print(f"{'Seg':<5}{'Trang':<12}{'Trang':<9}{'Predict':<25}{'Conf':<7}{'Match':<7}{'GT Label'}")
print("─" * 95)

correct = 0
results = []

for i, pages in enumerate(segments):
    page_start           = pages[0]["page_num"]
    page_end             = pages[-1]["page_num"]
    n_pages              = len(pages)
    label, conf, scores  = classify_segment(pages)
    gt_label             = GT_SEGMENT_TYPES.get(page_start, "?")
    is_correct           = (label == gt_label)
    if is_correct:
        correct += 1
    match = "✅" if is_correct else "❌"

    results.append({
        "segment"    : i + 1,
        "page_start" : page_start,
        "page_end"   : page_end,
        "n_pages"    : n_pages,
        "type"       : label,
        "confidence" : conf,
        "gt_type"    : gt_label,
        "match"      : is_correct,
        "scores"     : scores,
    })
    print(f"  {i+1:<4}{str(page_start)+'-'+str(page_end):<12}{n_pages:<9}"
          f"{label:<25}{conf:<7}{match}  {gt_label}")

print("=" * 95)
acc = correct / len(segments) if segments else 0
print(f"\n📊 Classification Accuracy: {correct}/{len(segments)} = {acc:.3f}  "
      f"{'✅ Đạt target 0.93' if acc >= 0.93 else '⚠️ Chưa đạt 0.93'}")

# Chi tiết segment sai
for r in results:
    if not r["match"]:
        i   = r["segment"] - 1
        hdr = segments[i][0]["header"][:100] if i < len(segments) else ""
        svb = re.findall(RE_SO_VAN_BAN, segments[i][0].get("header", "")) if i < len(segments) else []
        print(f"\n❌ Segment {r['segment']} (trang {r['page_start']}–{r['page_end']}):")
        print(f"   Predict : {r['type']} (conf={r['confidence']:.3f})")
        print(f"   GT      : {r['gt_type']}")
        print(f"   Scores  : {r['scores']}")
        print(f"   Số VB header : {svb}")
        print(f"   Header  : {hdr}")


SEGMENT CLASSIFICATION RESULTS
Seg  Trang       Trang    Predict                  Conf   Match  GT Label
───────────────────────────────────────────────────────────────────────────────────────────────
  1   1-3         3        Quyết định               0.596  ❌  Bản án hành chính
  2   4-6         3        Quyết định               0.707  ❌  ?
  3   7-8         2        Quyết định               0.554  ❌  ?
  4   9-28        20       Bản án dân sự            0.94   ❌  ?
  5   29-34       6        Bản án hình sự           0.947  ❌  Quyết định
  6   35-42       8        Bản án dân sự            0.911  ❌  Bản án hình sự
  7   43-43       1        Bản án hình sự           1.0    ❌  ?
  8   44-49       6        Bản án hình sự           0.979  ✅  Bản án hình sự
  9   50-51       2        Bản án dân sự            0.541  ❌  ?
  10  52-53       2        Quyết định               0.558  ❌  ?
  11  54-56       3        Quyết định               0.566  ❌  ?
  12  57-58       2        Quyết định       

In [17]:
# ── CELL 8: STAGE 4 — PDF SPLIT + JSON OUTPUT ────────────────
import uuid, time
from datetime import datetime
from pathlib import Path

OUTPUT_DIR = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/output'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


def split_pdf(pdf_path: str, segments: list, results: list, output_dir: str) -> dict:
    """
    Tách PDF gốc thành các file con theo segment.
    Xuất JSON result đúng contract spec.
    """
    src        = fitz.open(pdf_path)
    doc_id     = str(uuid.uuid4())[:8]
    start_time = time.time()
    sub_docs   = []
    warnings   = []

    for r in results:
        seg_idx    = r["segment"] - 1
        page_start = r["page_start"] - 1   # fitz dùng 0-indexed
        page_end   = r["page_end"] - 1

        # Tên file output
        label_safe = r["type"].replace(" ", "_")
        filename   = f"sub_{r['segment']:03d}_{label_safe}_p{r['page_start']}-{r['page_end']}.pdf"
        out_path   = f"{output_dir}/{filename}"

        # Tách và lưu file PDF con
        doc_out = fitz.open()
        doc_out.insert_pdf(src, from_page=page_start, to_page=page_end)
        doc_out.save(out_path)
        doc_out.close()

        sub_doc = {
            "sub_doc_id" : f"sub_{r['segment']:03d}",
            "type"       : r["type"],
            "page_start" : r["page_start"],
            "page_end"   : r["page_end"],
            "n_pages"    : r["n_pages"],
            "confidence" : r["confidence"],
            "output_file": filename,
        }
        sub_docs.append(sub_doc)

        # Warning nếu confidence thấp
        if r["confidence"] < 0.6:
            warnings.append({
                "sub_doc_id": f"sub_{r['segment']:03d}",
                "code"      : "LOW_CONFIDENCE_CLASSIFICATION",
                "message"   : f"Segment trang {r['page_start']}–{r['page_end']} có confidence thấp "
                              f"({r['confidence']:.2f}), đề nghị QA review.",
            })

    elapsed_ms = int((time.time() - start_time) * 1000)
    src.close()

    result_json = {
        "document_id"      : doc_id,
        "source_file"      : Path(pdf_path).name,
        "total_pages"      : TOTAL_PAGES,
        "total_segments"   : len(sub_docs),
        "processing_time_ms": elapsed_ms,
        "latency_per_page_ms": round(elapsed_ms / TOTAL_PAGES, 1),
        "created_at"       : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "sub_documents"    : sub_docs,
        "warnings"         : warnings,
    }

    # Lưu JSON
    json_path = f"{output_dir}/result_{doc_id}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result_json, f, ensure_ascii=False, indent=2)

    return result_json, json_path


# ── Chạy split ───────────────────────────────────────────────
print("⏳ Đang tách PDF...")
result_json, json_path = split_pdf(PDF_PATH, segments, results, OUTPUT_DIR)

# In kết quả
print("\nPDF SPLIT RESULTS")
print("=" * 70)
for sub in result_json["sub_documents"]:
    print(f"  [{sub['sub_doc_id']}] Trang {sub['page_start']:>2}–{sub['page_end']:>2} "
          f"({sub['n_pages']:>2} trang) | {sub['type']:<28} | conf={sub['confidence']:.2f}")
    print(f"          └─ {sub['output_file']}")

print("=" * 70)
print(f"\n📄 JSON output : {json_path}")
print(f"⏱ Thời gian   : {result_json['processing_time_ms']} ms "
      f"({result_json['latency_per_page_ms']} ms/trang)")
print(f"⚠ Warnings    : {len(result_json['warnings'])} segment cần QA review")

if result_json["warnings"]:
    for w in result_json["warnings"]:
        print(f"   → {w['message']}")

# In JSON mẫu để kiểm tra format
print("\nJSON OUTPUT (xem trước):")
print(json.dumps(result_json, ensure_ascii=False, indent=2)[:800] + "\n...")

# ── Verify: đọc lại từng file PDF con, in số trang + header ──
print("\nVERIFY OUTPUT FILES")
print("─" * 70)
for sub in result_json["sub_documents"]:
    path = f"{OUTPUT_DIR}/{sub['output_file']}"
    with fitz.open(path) as doc:
        n     = len(doc)
        head  = doc[0].get_text("text").strip()[:80].replace("\n", " ")
    status = "✅" if n == sub["n_pages"] else "❌ SỐ TRANG LỆCH"
    print(f"  {status} {sub['output_file']}")
    print(f"     Số trang: {n} | Header: {head}")

⏳ Đang tách PDF...

PDF SPLIT RESULTS
  [sub_001] Trang  1– 3 ( 3 trang) | Quyết định                   | conf=0.60
          └─ sub_001_Quyết_định_p1-3.pdf
  [sub_002] Trang  4– 6 ( 3 trang) | Quyết định                   | conf=0.71
          └─ sub_002_Quyết_định_p4-6.pdf
  [sub_003] Trang  7– 8 ( 2 trang) | Quyết định                   | conf=0.55
          └─ sub_003_Quyết_định_p7-8.pdf
  [sub_004] Trang  9–28 (20 trang) | Bản án dân sự                | conf=0.94
          └─ sub_004_Bản_án_dân_sự_p9-28.pdf
  [sub_005] Trang 29–34 ( 6 trang) | Bản án hình sự               | conf=0.95
          └─ sub_005_Bản_án_hình_sự_p29-34.pdf
  [sub_006] Trang 35–42 ( 8 trang) | Bản án dân sự                | conf=0.91
          └─ sub_006_Bản_án_dân_sự_p35-42.pdf
  [sub_007] Trang 43–43 ( 1 trang) | Bản án hình sự               | conf=1.00
          └─ sub_007_Bản_án_hình_sự_p43-43.pdf
  [sub_008] Trang 44–49 ( 6 trang) | Bản án hình sự               | conf=0.98
          └─ sub_008_Bản_án_hì

In [18]:
# ── CELL 9: WRAP full_pipeline() + ĐO LATENCY ────────────────
import time
from datetime import datetime

def full_pipeline(pdf_path: str, output_dir: str,
                  gt_boundaries: set = None,
                  gt_segment_types: dict = None) -> dict:
    """
    Chạy toàn bộ 4 stage: Extract → Boundary → Classify → Split
    Input : đường dẫn PDF
    Output: dict kết quả đầy đủ + metrics
    """
    t_total_start = time.time()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # ── STAGE 1: Extract Features ─────────────────────────────
    t1 = time.time()
    with fitz.open(pdf_path) as doc:
        n_pages = len(doc)

    features = []
    for i in range(n_pages):
        f = extract_features(pdf_path, i)
        # Cập nhật strict_header (tuple)
        f["strict_header"] = extract_strict_header(f["text_full"], n_lines=5)
        features.append(f)
    t1_ms = int((time.time() - t1) * 1000)

    # ── STAGE 2: Boundary Detection ───────────────────────────
    t2 = time.time()
    pred_boundaries = set()
    pred_separators = set()

    for i, feat in enumerate(features):
        prev = features[i-1] if i > 0 else None
        bdry, _, _ = is_boundary(feat, prev)
        if bdry:
            pred_boundaries.add(feat["page_num"])
        if feat["is_blank"]:
            pred_separators.add(feat["page_num"])
    t2_ms = int((time.time() - t2) * 1000)

    # Tính boundary metrics nếu có ground truth
    boundary_metrics = {}
    if gt_boundaries:
        TP = len(pred_boundaries & gt_boundaries)
        FP = len(pred_boundaries - gt_boundaries)
        FN = len(gt_boundaries - pred_boundaries)
        p  = TP / (TP + FP) if (TP + FP) > 0 else 0
        r  = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1 = 2*p*r/(p+r)    if (p + r)  > 0 else 0
        boundary_metrics = {
            "f1": round(f1, 3), "precision": round(p, 3),
            "recall": round(r, 3), "TP": TP, "FP": FP, "FN": FN
        }

    # ── STAGE 3: Segment Classification ───────────────────────
    t3 = time.time()
    segs    = group_segments(features, pred_boundaries, pred_separators)
    cls_results = []
    correct = 0

    for i, pages in enumerate(segs):
        page_start          = pages[0]["page_num"]
        page_end            = pages[-1]["page_num"]
        label, conf, scores = classify_segment(pages)
        gt_label            = gt_segment_types.get(page_start, "?") if gt_segment_types else "?"
        is_correct          = (label == gt_label) if gt_label != "?" else None
        if is_correct:
            correct += 1

        cls_results.append({
            "segment"    : i + 1,
            "page_start" : page_start,
            "page_end"   : page_end,
            "n_pages"    : len(pages),
            "type"       : label,
            "confidence" : conf,
            "gt_type"    : gt_label,
            "match"      : is_correct,
            "scores"     : scores,
        })
    t3_ms = int((time.time() - t3) * 1000)

    n_gt_known = sum(1 for r in cls_results if r["match"] is not None)
    cls_acc    = round(correct / n_gt_known, 3) if n_gt_known > 0 else 0

    # ── STAGE 4: PDF Split + JSON Output ──────────────────────
    t4 = time.time()
    src        = fitz.open(pdf_path)
    doc_id     = str(uuid.uuid4())[:8]
    sub_docs   = []
    warnings   = []

    for r in cls_results:
        label_safe = r["type"].replace(" ", "_")
        filename   = f"sub_{r['segment']:03d}_{label_safe}_p{r['page_start']}-{r['page_end']}.pdf"
        out_path   = f"{output_dir}/{filename}"

        doc_out = fitz.open()
        doc_out.insert_pdf(src,
                           from_page=r["page_start"] - 1,
                           to_page=r["page_end"] - 1)
        doc_out.save(out_path)
        doc_out.close()

        sub_docs.append({
            "sub_doc_id" : f"sub_{r['segment']:03d}",
            "type"       : r["type"],
            "page_start" : r["page_start"],
            "page_end"   : r["page_end"],
            "n_pages"    : r["n_pages"],
            "confidence" : r["confidence"],
            "output_file": filename,
        })
        if r["confidence"] < 0.6:
            warnings.append({
                "sub_doc_id": f"sub_{r['segment']:03d}",
                "code"      : "LOW_CONFIDENCE_CLASSIFICATION",
                "message"   : f"Trang {r['page_start']}–{r['page_end']} "
                              f"confidence={r['confidence']:.2f} — cần QA review.",
            })
    src.close()
    t4_ms = int((time.time() - t4) * 1000)

    # ── Tổng hợp output ───────────────────────────────────────
    t_total_ms = int((time.time() - t_total_start) * 1000)

    result = {
        "document_id"         : doc_id,
        "source_file"         : Path(pdf_path).name,
        "total_pages"         : n_pages,
        "total_segments"      : len(sub_docs),
        "output_dir"          : output_dir,
        "created_at"          : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "latency": {
            "stage1_extract_ms"  : t1_ms,
            "stage2_boundary_ms" : t2_ms,
            "stage3_classify_ms" : t3_ms,
            "stage4_split_ms"    : t4_ms,
            "total_ms"           : t_total_ms,
            "per_page_ms"        : round(t_total_ms / n_pages, 1),
            "sla_pass"           : t_total_ms / n_pages < 5000,   # SLA: <5s/trang
        },
        "metrics": {
            "boundary"          : boundary_metrics,
            "classification_accuracy": cls_acc,
            "classification_pass"    : cls_acc >= 0.93,
        },
        "sub_documents"       : sub_docs,
        "warnings"            : warnings,
    }

    # Lưu JSON
    json_path = f"{output_dir}/result_{doc_id}.json"
    with open(json_path, "w", encoding="utf-8") as fj:
        json.dump(result, fj, ensure_ascii=False, indent=2)
    result["json_path"] = json_path

    return result


# ── Chạy full_pipeline ────────────────────────────────────────
print("⏳ Chạy full_pipeline()...\n")

pipeline_result = full_pipeline(
    pdf_path         = PDF_PATH,
    output_dir       = OUTPUT_DIR,
    gt_boundaries    = GT_BOUNDARIES,
    gt_segment_types = GT_SEGMENT_TYPES,
)

# ── In kết quả ───────────────────────────────────────────────
r = pipeline_result

print("=" * 65)
print("FULL PIPELINE RESULTS")
print("=" * 65)
print(f"  File          : {r['source_file']}")
print(f"  Tổng trang    : {r['total_pages']}")
print(f"  Tổng segment  : {r['total_segments']}")
print(f"  JSON output   : {r['json_path']}")

print(f"\n⏱  LATENCY")
lat = r["latency"]
print(f"  Stage 1 Extract   : {lat['stage1_extract_ms']:>6} ms")
print(f"  Stage 2 Boundary  : {lat['stage2_boundary_ms']:>6} ms")
print(f"  Stage 3 Classify  : {lat['stage3_classify_ms']:>6} ms")
print(f"  Stage 4 Split     : {lat['stage4_split_ms']:>6} ms")
print(f"  ─────────────────────────────")
print(f"  Tổng              : {lat['total_ms']:>6} ms")
print(f"  Per page          : {lat['per_page_ms']:>6} ms/trang")
sla = "✅ Đạt (<5000ms)" if lat["sla_pass"] else "❌ Vượt SLA"
print(f"  SLA (<5s/trang)   : {sla}")

print(f"\n📊 METRICS")
bm = r["metrics"]["boundary"]
if bm:
    print(f"  Boundary F1       : {bm['f1']:.3f}  "
          f"{'✅' if bm['f1'] >= 0.9 else '⚠️'}")
    print(f"  Precision/Recall  : {bm['precision']:.3f} / {bm['recall']:.3f}")
acc = r["metrics"]["classification_accuracy"]
print(f"  Classification Acc: {acc:.3f}  "
      f"{'✅ Đạt 0.93' if r['metrics']['classification_pass'] else '⚠️ Chưa đạt 0.93'}")

print(f"\n📄 SUB DOCUMENTS")
for sub in r["sub_documents"]:
    conf_tag = "⚠️" if sub["confidence"] < 0.6 else "  "
    print(f"  {conf_tag} [{sub['sub_doc_id']}] "
          f"Trang {sub['page_start']:>2}–{sub['page_end']:>2} "
          f"({sub['n_pages']:>2} trang) | "
          f"{sub['type']:<25} | conf={sub['confidence']:.2f}")
    print(f"       └─ {sub['output_file']}")

if r["warnings"]:
    print(f"\n⚠️  WARNINGS ({len(r['warnings'])} segment cần QA review)")
    for w in r["warnings"]:
        print(f"   → {w['message']}")

print("=" * 65)
print(f"✅ Pipeline hoàn tất lúc {r['created_at']}")

⏳ Chạy full_pipeline()...

FULL PIPELINE RESULTS
  File          : document_merged2.pdf
  Tổng trang    : 109
  Tổng segment  : 21
  JSON output   : /content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/output/result_87081a38.json

⏱  LATENCY
  Stage 1 Extract   : 233242 ms
  Stage 2 Boundary  :      1 ms
  Stage 3 Classify  :      8 ms
  Stage 4 Split     :    652 ms
  ─────────────────────────────
  Tổng              : 233906 ms
  Per page          : 2145.9 ms/trang
  SLA (<5s/trang)   : ✅ Đạt (<5000ms)

📊 METRICS
  Boundary F1       : 1.000  ✅
  Precision/Recall  : 1.000 / 1.000
  Classification Acc: 0.250  ⚠️ Chưa đạt 0.93

📄 SUB DOCUMENTS
  ⚠️ [sub_001] Trang  1– 3 ( 3 trang) | Quyết định                | conf=0.60
       └─ sub_001_Quyết_định_p1-3.pdf
     [sub_002] Trang  4– 6 ( 3 trang) | Quyết định                | conf=0.71
       └─ sub_002_Quyết_định_p4-6.pdf
  ⚠️ [sub_003] Trang  7– 8 ( 2 trang) | Quyết định                | conf=0.55
       └─ sub_00